# Recovery improvement modeling notebook

This notebook now explores both:
- **recovery score regression** (exact percentage)
- **recovery zone classification** (red / orange / green)

It extends the first pass with a much richer behavioral feature set across Tier 1, Tier 2, and Tier 3 features.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "whoopdata").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from whoopdata.analytics.data_prep import get_recovery_modeling_dataset
from whoopdata.database.database import DB_PATH, SessionLocal

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid")
try:
    shap.initjs()
except Exception:
    pass


In [ ]:
print({"db_path": str(DB_PATH), "db_exists": DB_PATH.exists()})
db = SessionLocal()
try:
    recovery_df = get_recovery_modeling_dataset(db, days_back=None)
finally:
    db.close()

recovery_df = recovery_df.copy()
recovery_df["created_at"] = pd.to_datetime(recovery_df["created_at"])
recovery_df = recovery_df.sort_values("created_at").reset_index(drop=True)
print({"rows": len(recovery_df), "columns": len(recovery_df.columns)})
recovery_df[["created_at", "recovery_score", "sleep_hours", "strain", "hrv_rmssd_milli", "resting_heart_rate"]].tail()


In [ ]:
MINUTES_PER_DAY = 24 * 60

def datetime_to_clock_minutes(series: pd.Series) -> pd.Series:
    ts = pd.to_datetime(series)
    return ts.dt.hour * 60 + ts.dt.minute

def circular_minute_delta(current: pd.Series, reference: pd.Series) -> pd.Series:
    raw = current - reference
    return ((raw + MINUTES_PER_DAY / 2) % MINUTES_PER_DAY) - (MINUTES_PER_DAY / 2)

def circular_distance_minutes(current: pd.Series, reference: pd.Series) -> pd.Series:
    return circular_minute_delta(current, reference).abs()

def circular_rolling_std(series: pd.Series, window: int) -> pd.Series:
    radians = (series / MINUTES_PER_DAY) * 2 * np.pi
    min_periods = max(2, window // 2)
    sin_mean = np.sin(radians).shift(1).rolling(window, min_periods=min_periods).mean()
    cos_mean = np.cos(radians).shift(1).rolling(window, min_periods=min_periods).mean()
    resultant = np.sqrt(sin_mean**2 + cos_mean**2).clip(lower=1e-9, upper=1)
    circ_std = np.sqrt(-2 * np.log(resultant))
    return circ_std * (MINUTES_PER_DAY / (2 * np.pi))

def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    out = a / b.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)

def build_recovery_zone(score: pd.Series) -> pd.Series:
    return pd.Series(
        np.select(
            [score < 34, score < 67],
            ["red", "orange"],
            default="green",
        ),
        index=score.index,
    )

def add_recovery_improvement_features(df: pd.DataFrame) -> pd.DataFrame:
    engineered = df.copy()
    engineered = engineered.sort_values("created_at").reset_index(drop=True)

    engineered["sleep_start_clock_min"] = datetime_to_clock_minutes(engineered["sleep_start"])
    engineered["sleep_end_clock_min"] = datetime_to_clock_minutes(engineered["sleep_end"])
    engineered["sleep_midpoint_clock_min"] = (
        engineered["sleep_start_clock_min"] + (engineered["sleep_hours"] * 60 / 2)
    ) % MINUTES_PER_DAY

    engineered["workout_start_hour_numeric"] = pd.to_numeric(engineered["workout_start_hour"], errors="coerce")
    engineered["bedtime_hour_float"] = engineered["sleep_start_clock_min"] / 60.0
    engineered["wake_hour_float"] = engineered["sleep_end_clock_min"] / 60.0
    engineered["sleep_midpoint_hour_float"] = engineered["sleep_midpoint_clock_min"] / 60.0

    # Tier 1: timing deltas and variability
    for base_name, source_col in [
        ("bedtime", "sleep_start_clock_min"),
        ("wake_time", "sleep_end_clock_min"),
        ("sleep_midpoint", "sleep_midpoint_clock_min"),
    ]:
        engineered[f"{base_name}_delta_prev_day_min"] = circular_minute_delta(
            engineered[source_col], engineered[source_col].shift(1)
        )
        engineered[f"abs_{base_name}_delta_prev_day_min"] = engineered[f"{base_name}_delta_prev_day_min"].abs()
        for window in (3, 7):
            baseline = engineered[source_col].shift(1).rolling(window, min_periods=2).mean()
            engineered[f"{base_name}_delta_{window}d_avg_min"] = circular_minute_delta(engineered[source_col], baseline)
            engineered[f"abs_{base_name}_delta_{window}d_avg_min"] = circular_distance_minutes(engineered[source_col], baseline)
            engineered[f"{base_name}_variability_{window}d_min"] = circular_rolling_std(engineered[source_col], window)

    for window in (3, 7):
        engineered[f"sleep_hours_variability_{window}d"] = engineered["sleep_hours"].shift(1).rolling(window, min_periods=2).std()
        engineered[f"sleep_deficit_rolling_{window}d"] = engineered["sleep_deficit"].shift(1).rolling(window, min_periods=2).mean()
        engineered[f"sleep_deficit_sum_{window}d"] = engineered["sleep_deficit"].shift(1).rolling(window, min_periods=1).sum()

    engineered["late_bedtime_after_23"] = (engineered["sleep_start_clock_min"] >= 23 * 60).astype(int)
    engineered["late_bedtime_after_midnight"] = (engineered["sleep_start_clock_min"] < 5 * 60).astype(int)
    engineered["bedtime_shift_gt_60m_7d"] = (engineered["abs_bedtime_delta_7d_avg_min"] > 60).astype(int)
    engineered["wake_shift_gt_60m_7d"] = (engineered["abs_wake_time_delta_7d_avg_min"] > 60).astype(int)

    # Tier 1: sleep need and adequacy
    engineered["sleep_need_met"] = (engineered["sleep_deficit"] <= 0).astype(int)
    engineered["sleep_achieved_to_needed_ratio"] = safe_divide(engineered["sleep_hours"], engineered["total_sleep_need_hours"])
    engineered["sleep_achieved_to_baseline_need_ratio"] = safe_divide(engineered["sleep_hours"], engineered["baseline_sleep_needed_hours"])
    engineered["sleep_deficit_abs"] = engineered["sleep_deficit"].abs()
    engineered["mild_sleep_deficit_flag"] = ((engineered["sleep_deficit"] > 0.25) & (engineered["sleep_deficit"] <= 0.75)).astype(int)
    engineered["moderate_sleep_deficit_flag"] = ((engineered["sleep_deficit"] > 0.75) & (engineered["sleep_deficit"] <= 1.5)).astype(int)
    engineered["severe_sleep_deficit_flag"] = (engineered["sleep_deficit"] > 1.5).astype(int)

    consecutive_deficit = []
    run = 0
    for val in engineered["sleep_deficit"].fillna(0):
        run = run + 1 if val > 0 else 0
        consecutive_deficit.append(run)
    engineered["consecutive_sleep_deficit_days"] = consecutive_deficit

    # Tier 2: sleep architecture and fragmentation
    engineered["restorative_sleep_hours"] = engineered["rem_sleep_hours"] + engineered["slow_wave_sleep_hours"]
    engineered["restorative_sleep_ratio"] = safe_divide(engineered["restorative_sleep_hours"], engineered["sleep_hours"])
    engineered["rem_to_deep_ratio"] = safe_divide(engineered["rem_sleep_hours"], engineered["slow_wave_sleep_hours"])
    engineered["disturbances_per_hour"] = safe_divide(engineered["disturbance_count"], engineered["sleep_hours"])
    engineered["awake_fraction_of_time_in_bed"] = safe_divide(engineered["awake_time_hours"], engineered["time_in_bed_hours"])
    for metric in ["rem_sleep_hours", "slow_wave_sleep_hours", "awake_time_hours", "disturbance_count", "restorative_sleep_ratio"]:
        engineered[f"{metric}_rolling_7d"] = engineered[metric].shift(1).rolling(7, min_periods=3).mean()
        engineered[f"{metric}_vs_7d"] = engineered[metric] - engineered[f"{metric}_rolling_7d"]

    # Tier 2: social jetlag and weekday/weekend structure
    engineered["is_weekday"] = (~engineered["is_weekend"]).astype(int)
    engineered["weekday_bedtime_rolling_7d"] = engineered["sleep_start_clock_min"].where(~engineered["is_weekend"]).shift(1).rolling(7, min_periods=2).mean()
    engineered["weekend_bedtime_rolling_7d"] = engineered["sleep_start_clock_min"].where(engineered["is_weekend"]).shift(1).rolling(7, min_periods=1).mean()
    engineered["weekday_wake_rolling_7d"] = engineered["sleep_end_clock_min"].where(~engineered["is_weekend"]).shift(1).rolling(7, min_periods=2).mean()
    engineered["weekend_wake_rolling_7d"] = engineered["sleep_end_clock_min"].where(engineered["is_weekend"]).shift(1).rolling(7, min_periods=1).mean()
    engineered["social_jetlag_bedtime_gap_min"] = (engineered["weekend_bedtime_rolling_7d"] - engineered["weekday_bedtime_rolling_7d"]).abs()
    engineered["social_jetlag_wake_gap_min"] = (engineered["weekend_wake_rolling_7d"] - engineered["weekday_wake_rolling_7d"]).abs()

    # Tier 1/2: training load context
    engineered["strain_2d_sum"] = engineered["strain"].shift(1).rolling(2, min_periods=1).sum()
    engineered["strain_7d_sum"] = engineered["strain"].shift(1).rolling(7, min_periods=3).sum()
    engineered["strain_std_7d"] = engineered["strain"].shift(1).rolling(7, min_periods=3).std()
    engineered["strain_monotony_7d"] = safe_divide(engineered["strain_rolling_7d"], engineered["strain_std_7d"])
    engineered["strain_14d_sum"] = engineered["strain"].shift(1).rolling(14, min_periods=5).sum()
    engineered["acute_chronic_strain_ratio"] = safe_divide(engineered["strain_3d_sum"], safe_divide(engineered["strain_14d_sum"], pd.Series(14/3, index=engineered.index)))
    engineered["high_strain_day"] = (engineered["strain"] >= 14).astype(int)
    engineered["high_strain_prev_day"] = (engineered["prev_strain"] >= 14).astype(int)
    engineered["hard_day_after_hard_day"] = ((engineered["strain"] >= 14) & (engineered["prev_strain"] >= 14)).astype(int)

    # Tier 1/2: workout composition and timing
    engineered["zone_4_5_minutes"] = engineered["zone_four_minutes"] + engineered["zone_five_minutes"]
    engineered["zone_0_2_minutes"] = engineered["zone_zero_minutes"] + engineered["zone_one_minutes"] + engineered["zone_two_minutes"]
    engineered["late_workout_flag"] = ((engineered["workout_start_hour_numeric"] >= 19).fillna(False)).astype(int)
    engineered["has_workout_flag"] = (engineered["total_zone_minutes"] > 0).astype(int)
    engineered["high_intensity_workout_flag"] = (engineered["high_intensity_pct"] >= 20).astype(int)

    sport_group = engineered["sport_id"].replace(0, np.nan)
    engineered["running_like_sport"] = sport_group.isin([1, 16]).fillna(False).astype(int)
    engineered["tennis_like_sport"] = sport_group.isin([43]).fillna(False).astype(int)

    days_since_hi = []
    last_hi_idx = None
    for idx, flag in enumerate(engineered["high_intensity_workout_flag"].shift(1).fillna(0)):
        if flag == 1:
            last_hi_idx = idx
        days_since_hi.append(np.nan if last_hi_idx is None else idx - last_hi_idx)
    engineered["days_since_high_intensity"] = days_since_hi

    # Tier 1/2: interaction features
    engineered["sleep_deficit_x_strain"] = engineered["sleep_deficit"] * engineered["strain"]
    engineered["bedtime_irregularity_x_strain"] = engineered["abs_bedtime_delta_7d_avg_min"] * engineered["strain"]
    engineered["late_workout_x_bedtime_shift"] = engineered["late_workout_flag"] * engineered["abs_bedtime_delta_prev_day_min"]
    engineered["high_strain_and_short_sleep"] = ((engineered["strain"] >= 14) & (engineered["sleep_hours"] < 7)).astype(int)
    engineered["high_strain_and_irregular_bedtime"] = ((engineered["strain"] >= 14) & (engineered["abs_bedtime_delta_7d_avg_min"] > 60)).astype(int)

    # Tier 3: adherence / habit pattern features
    engineered["sleep_need_met_7d_rate"] = engineered["sleep_need_met"].shift(1).rolling(7, min_periods=3).mean()
    engineered["late_bedtime_7d_rate"] = engineered["late_bedtime_after_23"].shift(1).rolling(7, min_periods=3).mean()
    engineered["bedtime_shift_gt_60m_7d_rate"] = engineered["bedtime_shift_gt_60m_7d"].shift(1).rolling(7, min_periods=3).mean()
    engineered["high_strain_7d_rate"] = engineered["high_strain_day"].shift(1).rolling(7, min_periods=3).mean()
    engineered["green_day_flag"] = (engineered["recovery_score"] >= 67).astype(int)
    engineered["red_day_flag"] = (engineered["recovery_score"] < 34).astype(int)

    engineered["recovery_zone"] = build_recovery_zone(engineered["recovery_score"])
    return engineered

feature_df = add_recovery_improvement_features(recovery_df)
feature_df[["created_at", "recovery_score", "recovery_zone", "sleep_hours", "strain", "abs_bedtime_delta_7d_avg_min", "sleep_achieved_to_needed_ratio", "zone_4_5_minutes"]].tail(12)


In [ ]:
coverage_columns = [
    "recovery_score", "recovery_zone", "strain", "sleep_hours", "sleep_efficiency_percentage",
    "sleep_consistency_percentage", "sleep_performance_percentage", "sleep_start", "sleep_end",
    "sleep_deficit", "abs_bedtime_delta_7d_avg_min", "zone_4_5_minutes", "sleep_need_met_7d_rate"
]
feature_df[coverage_columns].isna().mean().sort_values().to_frame("missing_fraction")


In [ ]:
candidate_features = [
    # Core physiology
    "hrv_rmssd_milli", "prev_hrv", "hrv_rolling_7d", "hrv_deviation_from_avg", "hrv_std_7d",
    "resting_heart_rate", "prev_rhr", "rhr_rolling_7d", "rhr_deviation_from_avg", "rhr_std_7d",
    "prev_recovery_score", "recovery_rolling_7d", "recovery_std_7d",
    # Sleep need / adequacy
    "sleep_hours", "time_in_bed_hours", "sleep_efficiency_percentage", "sleep_consistency_percentage",
    "sleep_performance_percentage", "sleep_deficit", "sleep_deficit_abs", "sleep_achieved_to_needed_ratio",
    "sleep_achieved_to_baseline_need_ratio", "baseline_sleep_needed_hours", "sleep_debt_hours",
    "sleep_need_from_strain_hours", "total_sleep_need_hours", "sleep_need_met",
    "mild_sleep_deficit_flag", "moderate_sleep_deficit_flag", "severe_sleep_deficit_flag",
    "consecutive_sleep_deficit_days", "sleep_deficit_rolling_3d", "sleep_deficit_rolling_7d",
    "sleep_deficit_sum_3d", "sleep_deficit_sum_7d",
    # Timing / regularity
    "sleep_start_clock_min", "sleep_end_clock_min", "sleep_midpoint_clock_min",
    "bedtime_hour_float", "wake_hour_float", "sleep_midpoint_hour_float",
    "bedtime_delta_prev_day_min", "wake_time_delta_prev_day_min", "sleep_midpoint_delta_prev_day_min",
    "abs_bedtime_delta_prev_day_min", "abs_wake_time_delta_prev_day_min", "abs_sleep_midpoint_delta_prev_day_min",
    "bedtime_delta_3d_avg_min", "wake_time_delta_3d_avg_min", "sleep_midpoint_delta_3d_avg_min",
    "bedtime_delta_7d_avg_min", "wake_time_delta_7d_avg_min", "sleep_midpoint_delta_7d_avg_min",
    "abs_bedtime_delta_3d_avg_min", "abs_wake_time_delta_3d_avg_min", "abs_sleep_midpoint_delta_3d_avg_min",
    "abs_bedtime_delta_7d_avg_min", "abs_wake_time_delta_7d_avg_min", "abs_sleep_midpoint_delta_7d_avg_min",
    "bedtime_variability_3d_min", "wake_time_variability_3d_min", "sleep_midpoint_variability_3d_min",
    "bedtime_variability_7d_min", "wake_time_variability_7d_min", "sleep_midpoint_variability_7d_min",
    "sleep_hours_variability_3d", "sleep_hours_variability_7d",
    "late_bedtime_after_23", "late_bedtime_after_midnight", "bedtime_shift_gt_60m_7d", "wake_shift_gt_60m_7d",
    # Sleep architecture
    "rem_sleep_hours", "slow_wave_sleep_hours", "light_sleep_hours", "awake_time_hours",
    "rem_percentage", "deep_sleep_percentage", "light_sleep_percentage",
    "restorative_sleep_hours", "restorative_sleep_ratio", "rem_to_deep_ratio",
    "disturbance_count", "disturbances_per_hour", "awake_fraction_of_time_in_bed",
    "sleep_cycle_count", "respiratory_rate",
    "rem_sleep_hours_vs_7d", "slow_wave_sleep_hours_vs_7d", "awake_time_hours_vs_7d",
    "disturbance_count_vs_7d", "restorative_sleep_ratio_vs_7d",
    # Training load
    "strain", "prev_strain", "strain_2d_sum", "strain_3d_sum", "strain_7d_sum",
    "strain_rolling_7d", "strain_14d_sum", "strain_std_7d", "strain_monotony_7d",
    "strain_deviation_from_avg", "acute_chronic_strain_ratio",
    "high_strain_day", "high_strain_prev_day", "hard_day_after_hard_day",
    # Workout composition
    "workout_strain", "workout_kilojoule", "total_zone_minutes", "high_intensity_pct",
    "zone_4_5_minutes", "zone_0_2_minutes",
    "zone_zero_minutes", "zone_one_minutes", "zone_two_minutes", "zone_three_minutes", "zone_four_minutes", "zone_five_minutes",
    "zone_zero_pct", "zone_one_pct", "zone_two_pct", "zone_three_pct", "zone_four_pct", "zone_five_pct",
    "workout_start_hour", "workout_is_morning", "workout_is_afternoon", "workout_is_evening",
    "late_workout_flag", "has_workout_flag", "high_intensity_workout_flag", "days_since_high_intensity",
    "running_like_sport", "tennis_like_sport",
    # Calendar / habit patterns
    "day_of_week", "is_weekend", "is_weekday",
    "social_jetlag_bedtime_gap_min", "social_jetlag_wake_gap_min",
    "sleep_need_met_7d_rate", "late_bedtime_7d_rate", "bedtime_shift_gt_60m_7d_rate", "high_strain_7d_rate",
    # Interactions
    "sleep_deficit_x_strain", "bedtime_irregularity_x_strain", "late_workout_x_bedtime_shift",
    "high_strain_and_short_sleep", "high_strain_and_irregular_bedtime",
]

available_features = [c for c in candidate_features if c in feature_df.columns]
model_df = feature_df.copy()
model_df = model_df.dropna(subset=["recovery_score", "recovery_zone"]).copy()
feature_missing_fraction = model_df[available_features].isna().mean().sort_values()
selected_features = feature_missing_fraction[feature_missing_fraction <= 0.35].index.tolist()

for col in selected_features:
    if pd.api.types.is_bool_dtype(model_df[col]):
        model_df[col] = model_df[col].fillna(False).astype(int)
    else:
        model_df[col] = model_df[col].fillna(model_df[col].median())

print({
    "candidate_features": len(candidate_features),
    "available_features": len(available_features),
    "selected_features": len(selected_features),
    "rows_after_filtering": len(model_df),
})
feature_missing_fraction.to_frame("missing_fraction").head(40)


In [ ]:
X = model_df[selected_features]
y_reg = model_df["recovery_score"]
y_cls = model_df["recovery_zone"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

reg_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
reg_model.fit(X_train_reg, y_train_reg)
reg_preds = reg_model.predict(X_test_reg)

reg_metrics = pd.DataFrame([
    {"metric": "rows_train", "value": len(X_train_reg)},
    {"metric": "rows_test", "value": len(X_test_reg)},
    {"metric": "mae", "value": mean_absolute_error(y_test_reg, reg_preds)},
    {"metric": "rmse", "value": root_mean_squared_error(y_test_reg, reg_preds)},
    {"metric": "r2", "value": r2_score(y_test_reg, reg_preds)},
]).set_index("metric")
reg_metrics


In [ ]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

cls_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=3,
    max_features=0.35,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
cls_model.fit(X_train_cls, y_train_cls)
cls_preds = cls_model.predict(X_test_cls)

cls_metrics = pd.DataFrame([
    {"metric": "rows_train", "value": len(X_train_cls)},
    {"metric": "rows_test", "value": len(X_test_cls)},
    {"metric": "accuracy", "value": accuracy_score(y_test_cls, cls_preds)},
    {"metric": "macro_f1", "value": f1_score(y_test_cls, cls_preds, average="macro")},
]).set_index("metric")
cls_metrics


In [ ]:
prediction_frame = pd.DataFrame({
    "actual_recovery": y_test_reg.reset_index(drop=True),
    "predicted_recovery": pd.Series(reg_preds),
})
prediction_frame["error"] = prediction_frame["predicted_recovery"] - prediction_frame["actual_recovery"]
prediction_frame.head(15)


In [ ]:
cm = pd.DataFrame(
    confusion_matrix(y_test_cls, cls_preds, labels=["red", "orange", "green"]),
    index=["actual_red", "actual_orange", "actual_green"],
    columns=["pred_red", "pred_orange", "pred_green"],
)
cm


In [ ]:
print(classification_report(y_test_cls, cls_preds, labels=["red", "orange", "green"]))


In [ ]:
reg_explainer = shap.TreeExplainer(reg_model)
reg_shap_values = reg_explainer(X_test_reg)
reg_shap_importance = (
    pd.DataFrame({
        "feature": X_test_reg.columns,
        "mean_abs_shap": np.abs(reg_shap_values.values).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
reg_shap_importance.head(25)


In [ ]:
plt.figure(figsize=(10, 8))
top_features = reg_shap_importance.head(20).sort_values("mean_abs_shap")
plt.barh(top_features["feature"], top_features["mean_abs_shap"])
plt.title("Regression: top features by mean absolute SHAP value")
plt.xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(reg_shap_values.values, X_test_reg, max_display=20, show=False)
plt.tight_layout()
plt.show()


In [ ]:
top_reg_feature = reg_shap_importance.iloc[0]["feature"]
shap.dependence_plot(top_reg_feature, reg_shap_values.values, X_test_reg, display_features=X_test_reg)


In [ ]:
cls_importance = pd.DataFrame({
    "feature": X_train_cls.columns,
    "importance": cls_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
cls_importance.head(25)


In [ ]:
class_balance = model_df["recovery_zone"].value_counts().rename_axis("zone").to_frame("count")
class_balance["share"] = class_balance["count"] / class_balance["count"].sum()
class_balance


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Classification confusion matrix")
plt.tight_layout()
plt.show()


In [ ]:
cls_proba = pd.DataFrame(
    cls_model.predict_proba(X_test_cls),
    columns=[f"proba_{label}" for label in cls_model.classes_],
    index=X_test_cls.index,
)
cls_prediction_frame = X_test_cls.copy()
cls_prediction_frame["actual_zone"] = y_test_cls
cls_prediction_frame["predicted_zone"] = cls_preds
cls_prediction_frame = cls_prediction_frame.join(cls_proba)
cls_prediction_frame["max_predicted_proba"] = cls_proba.max(axis=1)
cls_prediction_frame["correct"] = (cls_prediction_frame["actual_zone"] == cls_prediction_frame["predicted_zone"])
cls_prediction_frame[["actual_zone", "predicted_zone", "proba_red", "proba_orange", "proba_green", "max_predicted_proba", "correct"]].head(20)


In [ ]:
cls_prediction_frame.sort_values(["correct", "max_predicted_proba"], ascending=[True, True])[
    ["actual_zone", "predicted_zone", "proba_red", "proba_orange", "proba_green", "max_predicted_proba"]
].head(15)


In [ ]:
cls_explainer = shap.TreeExplainer(cls_model)
cls_shap_values = cls_explainer(X_test_cls)

cls_shap_array = cls_shap_values.values
if cls_shap_array.ndim == 3:
    class_axis = 2
    cls_mean_abs_shap = np.abs(cls_shap_array).mean(axis=0)
    cls_shap_tables = {}
    for idx, label in enumerate(cls_model.classes_):
        cls_shap_tables[label] = (
            pd.DataFrame({
                "feature": X_test_cls.columns,
                "mean_abs_shap": cls_mean_abs_shap[:, idx],
            })
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )
else:
    cls_shap_tables = {label: cls_importance.copy() for label in cls_model.classes_}

cls_shap_tables["red"].head(20)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, label in zip(axes, ["red", "orange", "green"]):
    table = cls_shap_tables[label].head(12).sort_values("mean_abs_shap")
    ax.barh(table["feature"], table["mean_abs_shap"])
    ax.set_title(f"Classifier feature impact: {label}")
plt.tight_layout()
plt.show()


## Actionability analysis: personalized threshold-style coaching rules

This section shifts from pure prediction to simple, user-facing rules derived from historical patterns.
The goal is to identify concrete thresholds around bedtime, sleep, strain, and their interactions that are both interpretable and controllable.


In [ ]:
action_df = feature_df.copy()
action_df = action_df.sort_values("created_at").reset_index(drop=True)
action_df["green_flag"] = (action_df["recovery_zone"] == "green").astype(int)
action_df["red_or_orange_flag"] = (action_df["recovery_zone"] != "green").astype(int)
action_df["sleep_start_min_extended"] = action_df["sleep_start_clock_min"].where(action_df["sleep_start_clock_min"] >= 12 * 60, action_df["sleep_start_clock_min"] + 24 * 60)
action_df["bedtime_hour_extended"] = action_df["sleep_start_min_extended"] / 60.0
action_df["short_sleep_flag"] = (action_df["sleep_hours"] < 7.0).astype(int)
action_df["short_sleep_2plus"] = (action_df["consecutive_sleep_deficit_days"] >= 2).astype(int)
action_df["high_3d_strain_flag"] = 0
strain_threshold_candidates = sorted({round(x, 1) for x in action_df["strain_3d_sum"].dropna().quantile([0.5, 0.65, 0.75, 0.85]).tolist()})
action_df.head(3)


In [ ]:
def summarize_rule(df: pd.DataFrame, mask: pd.Series, rule_name: str, recommendation: str, min_group_size: int = 25) -> dict | None:
    mask = mask.fillna(False)
    in_group = df.loc[mask]
    out_group = df.loc[~mask]
    if len(in_group) < min_group_size or len(out_group) < min_group_size:
        return None
    return {
        "rule": rule_name,
        "recommendation": recommendation,
        "days_in_rule": int(len(in_group)),
        "days_outside_rule": int(len(out_group)),
        "avg_recovery_in_rule": float(in_group["recovery_score"].mean()),
        "avg_recovery_outside_rule": float(out_group["recovery_score"].mean()),
        "green_rate_in_rule": float(in_group["green_flag"].mean()),
        "green_rate_outside_rule": float(out_group["green_flag"].mean()),
        "recovery_delta": float(in_group["recovery_score"].mean() - out_group["recovery_score"].mean()),
        "green_rate_delta": float(in_group["green_flag"].mean() - out_group["green_flag"].mean()),
    }

def search_threshold_rule(df: pd.DataFrame, feature: str, thresholds: list[float], direction: str, label_fmt, recommendation_fmt, min_group_size: int = 25) -> pd.DataFrame:
    rows = []
    series = df[feature]
    for threshold in thresholds:
        if direction == "le":
            mask = series <= threshold
        elif direction == "ge":
            mask = series >= threshold
        else:
            raise ValueError(direction)
        summary = summarize_rule(df, mask, label_fmt(threshold), recommendation_fmt(threshold), min_group_size=min_group_size)
        if summary is not None:
            summary["feature"] = feature
            summary["threshold"] = float(threshold)
            rows.append(summary)
    return pd.DataFrame(rows)

bedtime_thresholds = [22.0, 22.5, 23.0, 23.5, 24.0, 24.5]
sleep_thresholds = [6.5, 7.0, 7.5, 8.0, 8.5]
short_sleep_streak_thresholds = [1, 2, 3]
strain_thresholds = strain_threshold_candidates

bedtime_rules = search_threshold_rule(
    action_df,
    feature="bedtime_hour_extended",
    thresholds=bedtime_thresholds,
    direction="le",
    label_fmt=lambda t: f"Bedtime at or before {int(t % 24):02d}:{int((t % 1) * 60):02d}",
    recommendation_fmt=lambda t: f"Aim to be asleep by about {int(t % 24):02d}:{int((t % 1) * 60):02d} or earlier",
)

sleep_rules = search_threshold_rule(
    action_df,
    feature="sleep_hours",
    thresholds=sleep_thresholds,
    direction="ge",
    label_fmt=lambda t: f"Sleep at least {t:.1f}h",
    recommendation_fmt=lambda t: f"Protect at least {t:.1f}h of sleep opportunity",
)

streak_rules = search_threshold_rule(
    action_df,
    feature="consecutive_sleep_deficit_days",
    thresholds=short_sleep_streak_thresholds,
    direction="le",
    label_fmt=lambda t: f"No more than {int(t)} consecutive sleep-deficit night(s)",
    recommendation_fmt=lambda t: f"Avoid letting sleep deficit extend beyond {int(t)} consecutive night(s)",
)

strain_rules = search_threshold_rule(
    action_df,
    feature="strain_3d_sum",
    thresholds=strain_thresholds,
    direction="le",
    label_fmt=lambda t: f"Keep 3-day strain at or below {t:.1f}",
    recommendation_fmt=lambda t: f"Keep recent 3-day cumulative strain around {t:.1f} or lower when possible",
)

candidate_rules = pd.concat([bedtime_rules, sleep_rules, streak_rules, strain_rules], ignore_index=True)
candidate_rules["score"] = candidate_rules["green_rate_delta"] * 100 + candidate_rules["recovery_delta"]
candidate_rules.sort_values(["score", "green_rate_delta", "recovery_delta"], ascending=False).head(15)


In [ ]:
best_rule_per_feature = (
    candidate_rules.sort_values(["feature", "score"], ascending=[True, False])
    .groupby("feature", as_index=False)
    .first()
    .sort_values("score", ascending=False)
)
best_rule_per_feature[[
    "rule", "recommendation", "days_in_rule", "avg_recovery_in_rule",
    "avg_recovery_outside_rule", "recovery_delta", "green_rate_in_rule",
    "green_rate_outside_rule", "green_rate_delta"
]]


In [ ]:
strain_cut = float(best_rule_per_feature.loc[best_rule_per_feature["feature"] == "strain_3d_sum", "threshold"].iloc[0]) if (best_rule_per_feature["feature"] == "strain_3d_sum").any() else float(action_df["strain_3d_sum"].median())
sleep_cut = float(best_rule_per_feature.loc[best_rule_per_feature["feature"] == "sleep_hours", "threshold"].iloc[0]) if (best_rule_per_feature["feature"] == "sleep_hours").any() else 7.5
bedtime_cut = float(best_rule_per_feature.loc[best_rule_per_feature["feature"] == "bedtime_hour_extended", "threshold"].iloc[0]) if (best_rule_per_feature["feature"] == "bedtime_hour_extended").any() else 23.5

interaction_conditions = {
    "high_strain_and_short_sleep": (action_df["strain_3d_sum"] >= strain_cut) & (action_df["sleep_hours"] < sleep_cut),
    "high_strain_and_late_bedtime": (action_df["strain_3d_sum"] >= strain_cut) & (action_df["bedtime_hour_extended"] > bedtime_cut),
    "high_strain_with_good_sleep": (action_df["strain_3d_sum"] >= strain_cut) & (action_df["sleep_hours"] >= sleep_cut),
}

interaction_rows = []
for name, mask in interaction_conditions.items():
    summary = summarize_rule(
        action_df,
        mask,
        rule_name=name,
        recommendation=f"Watch {name.replace('_', ' ')}",
        min_group_size=20,
    )
    if summary is not None:
        interaction_rows.append(summary)

interaction_rules = pd.DataFrame(interaction_rows).sort_values(["green_rate_delta", "recovery_delta"])
interaction_rules[[
    "rule", "days_in_rule", "avg_recovery_in_rule", "avg_recovery_outside_rule",
    "recovery_delta", "green_rate_in_rule", "green_rate_outside_rule", "green_rate_delta"
]]


In [ ]:
top_candidate_rules = best_rule_per_feature[[
    "rule", "recommendation", "days_in_rule", "recovery_delta", "green_rate_delta"
]].copy()
top_candidate_rules["green_rate_delta_pct_points"] = top_candidate_rules["green_rate_delta"] * 100
top_candidate_rules = top_candidate_rules.drop(columns=["green_rate_delta"]).sort_values("green_rate_delta_pct_points", ascending=False)
top_candidate_rules


### How to use these outputs

- Treat these as **candidate personalized rules**, not causal proof.
- Prefer thresholds with both a meaningful green-rate lift and enough historical support.
- The interaction table is often the most product-useful because it shows when strain is only costly in combination with poor sleep or later bedtime.


## What changed in this version

- Added Tier 1 features: timing irregularity, sleep need adequacy, training-load interactions
- Added Tier 2 features: sleep architecture quality, fragmentation, social jetlag, monotony/load patterns
- Added Tier 3 features: habit/adherence rates and broader contextual behavior summaries
- Added a classification path for **red / orange / green** recovery zones

## What to inspect next

- do the top features become more behaviorally interpretable?
- does classification give more user-friendly insight than raw-score regression?
- which features are predictive **and** controllable enough to support coaching?
